# 7. Introduction to Options - Interactive Explorer

**Objective**: Understand options, moneyness, and strategies by interactively adjusting parameters and seeing real-time payoff changes.

## What You'll Learn
- **Call Options**: The right to BUY at strike price
- **Put Options**: The right to SELL at strike price
- **Moneyness**: ITM, ATM, OTM classification
- **Protective Put**: Insurance strategy for stock protection
- **Risk Transformation**: How options change payoff profiles

Use the sliders below to explore how each parameter affects option profitability!

In [1]:
# Import Required Libraries
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    widgets_available = True
except ImportError:
    widgets_available = False
    print("⚠ ipywidgets not available. Please install: pip install ipywidgets")

# Set matplotlib to use inline display
%matplotlib inline

print("✓ Libraries loaded successfully")

✓ Libraries loaded successfully


In [2]:
def explore_options(current_price, strike_price, call_premium, put_premium, strategy):
    """
    Interactive options explorer.
    
    Parameters:
    - current_price: Current stock price (S0)
    - strike_price: Strike price (K)
    - call_premium: Cost to buy call option
    - put_premium: Cost to buy put option
    - strategy: 'Call', 'Put', or 'Protective Put'
    """
    
    # Generate stock prices at expiration
    price_range = np.linspace(current_price * 0.6, current_price * 1.6, 300)
    
    # Calculate option payoffs (intrinsic value only, no time value)
    call_payoff = np.maximum(price_range - strike_price, 0)
    put_payoff = np.maximum(strike_price - price_range, 0)
    
    # Calculate profits (payoff - premium paid)
    call_profit = call_payoff - call_premium
    put_profit = put_payoff - put_premium
    
    # Stock profit (for protective put)
    stock_profit = price_range - current_price
    
    # Create figure with subplots based on strategy
    if strategy == 'Call':
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # LEFT: Call payoff
        ax1.plot(price_range, call_payoff, linewidth=2.5, color='darkgreen', label='Intrinsic Value')
        ax1.plot(price_range, call_profit, linewidth=2.5, color='lime', label=f'Profit (P&L)')
        ax1.axhline(0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
        ax1.axvline(strike_price, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Strike K=${strike_price}')
        ax1.axvline(strike_price + call_premium, color='blue', linestyle=':', linewidth=1.5, alpha=0.7, label=f'Break-even=${strike_price + call_premium:.0f}')
        ax1.fill_between(price_range, 0, call_profit, where=(call_profit > 0), color='green', alpha=0.2, label='Profit Zone')
        ax1.fill_between(price_range, 0, call_profit, where=(call_profit <= 0), color='red', alpha=0.2, label='Loss Zone')
        ax1.set_title('Long Call Option Profit/Loss', fontsize=12, fontweight='bold')
        ax1.set_xlabel('Stock Price at Expiration', fontsize=11)
        ax1.set_ylabel('Profit ($)', fontsize=11)
        ax1.legend(loc='best', fontsize=9)
        ax1.grid(True, alpha=0.3)
        
        # RIGHT: Moneyness visualization
        moneyness_label = "ATM"
        if price_range[-1] > strike_price + 5:
            moneyness_label = "ITM (In-The-Money)"
        elif price_range[0] < strike_price - 5:
            moneyness_label = "OTM (Out-of-The-Money)"
        
        ax2.plot(price_range, call_profit, linewidth=2.5, color='black', label='Call Profit')
        ax2.axvline(strike_price, color='purple', linestyle='--', linewidth=2, alpha=0.8, label=f'Strike=${strike_price}')
        ax2.axvline(current_price, color='green', linestyle=':', linewidth=2, alpha=0.7, label=f'Current=${current_price}')
        
        # Shade moneyness regions
        ax2.fill_between(price_range, -10, call_profit, where=(price_range < strike_price), color='red', alpha=0.15, label='OTM')
        ax2.fill_between(price_range, -10, call_profit, where=(price_range >= strike_price), color='green', alpha=0.15, label='ITM')
        
        ax2.set_title('Call Option Moneyness', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Stock Price at Expiration', fontsize=11)
        ax2.set_ylabel('Profit ($)', fontsize=11)
        ax2.legend(loc='best', fontsize=9)
        ax2.grid(True, alpha=0.3)
        
    elif strategy == 'Put':
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # LEFT: Put payoff
        ax1.plot(price_range, put_payoff, linewidth=2.5, color='darkred', label='Intrinsic Value')
        ax1.plot(price_range, put_profit, linewidth=2.5, color='salmon', label='Profit (P&L)')
        ax1.axhline(0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
        ax1.axvline(strike_price, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Strike K=${strike_price}')
        ax1.axvline(strike_price - put_premium, color='blue', linestyle=':', linewidth=1.5, alpha=0.7, label=f'Break-even=${strike_price - put_premium:.0f}')
        ax1.fill_between(price_range, 0, put_profit, where=(put_profit > 0), color='green', alpha=0.2, label='Profit Zone')
        ax1.fill_between(price_range, 0, put_profit, where=(put_profit <= 0), color='red', alpha=0.2, label='Loss Zone')
        ax1.set_title('Long Put Option Profit/Loss', fontsize=12, fontweight='bold')
        ax1.set_xlabel('Stock Price at Expiration', fontsize=11)
        ax1.set_ylabel('Profit ($)', fontsize=11)
        ax1.legend(loc='best', fontsize=9)
        ax1.grid(True, alpha=0.3)
        
        # RIGHT: Moneyness visualization
        ax2.plot(price_range, put_profit, linewidth=2.5, color='black', label='Put Profit')
        ax2.axvline(strike_price, color='purple', linestyle='--', linewidth=2, alpha=0.8, label=f'Strike=${strike_price}')
        ax2.axvline(current_price, color='green', linestyle=':', linewidth=2, alpha=0.7, label=f'Current=${current_price}')
        
        # Shade moneyness regions
        ax2.fill_between(price_range, -10, put_profit, where=(price_range >= strike_price), color='red', alpha=0.15, label='OTM')
        ax2.fill_between(price_range, -10, put_profit, where=(price_range < strike_price), color='green', alpha=0.15, label='ITM')
        
        ax2.set_title('Put Option Moneyness', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Stock Price at Expiration', fontsize=11)
        ax2.set_ylabel('Profit ($)', fontsize=11)
        ax2.legend(loc='best', fontsize=9)
        ax2.grid(True, alpha=0.3)
        
    else:  # Protective Put
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # LEFT: Protective put strategy
        protective_profit = stock_profit + put_payoff - put_premium
        
        ax1.plot(price_range, stock_profit, linewidth=2.5, color='gray', linestyle='--', label='Stock Only', alpha=0.7)
        ax1.plot(price_range, protective_profit, linewidth=2.5, color='blue', label='Stock + Put (Protected)')
        ax1.axhline(0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
        ax1.axvline(current_price, color='green', linestyle=':', linewidth=2, alpha=0.7, label=f'Current=${current_price}')
        ax1.axvline(strike_price, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Put Strike=${strike_price}')
        
        max_loss = strike_price - current_price - put_premium
        ax1.fill_between(price_range, stock_profit, protective_profit, where=(price_range < strike_price), color='lightblue', alpha=0.3, label='Protected Zone')
        ax1.annotate(f'Max Loss: ${max_loss:.1f}', xy=(strike_price - 5, max_loss), xytext=(strike_price - 30, max_loss + 5),
                    arrowprops=dict(arrowstyle='->', color='red', lw=1.5), fontsize=10, color='red', fontweight='bold')
        
        ax1.set_title('Protective Put: Insurance Strategy', fontsize=12, fontweight='bold')
        ax1.set_xlabel('Stock Price at Expiration', fontsize=11)
        ax1.set_ylabel('Profit ($)', fontsize=11)
        ax1.legend(loc='best', fontsize=9)
        ax1.grid(True, alpha=0.3)
        
        # RIGHT: Risk profile comparison
        long_call_profit = np.maximum(price_range - strike_price, 0) - call_premium
        
        ax2.plot(price_range, protective_profit, linewidth=2.5, color='blue', label='Protective Put')
        ax2.plot(price_range, long_call_profit, linewidth=2.5, color='purple', linestyle='--', label='Long Call (for comparison)')
        ax2.axhline(0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
        ax2.axvline(current_price, color='green', linestyle=':', linewidth=2, alpha=0.7)
        
        ax2.set_title('Strategy Comparison: Protective Put vs Call', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Stock Price at Expiration', fontsize=11)
        ax2.set_ylabel('Profit ($)', fontsize=11)
        ax2.legend(loc='best', fontsize=9)
        ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print analysis
    print(f"\n📊 OPTION ANALYSIS")
    print(f"{'='*60}")
    if strategy == 'Call':
        breakeven = strike_price + call_premium
        print(f"Strategy: LONG CALL")
        print(f"Strike Price: ${strike_price:.0f}")
        print(f"Premium Paid: ${call_premium:.2f}")
        print(f"Break-even: ${breakeven:.2f}")
        print(f"Max Loss: ${call_premium:.2f} (premium paid)")
        print(f"Max Gain: Unlimited")
        print(f"Current Moneyness: {'ITM' if current_price > strike_price else 'ATM' if abs(current_price - strike_price) < 5 else 'OTM'}")
    elif strategy == 'Put':
        breakeven = strike_price - put_premium
        print(f"Strategy: LONG PUT")
        print(f"Strike Price: ${strike_price:.0f}")
        print(f"Premium Paid: ${put_premium:.2f}")
        print(f"Break-even: ${breakeven:.2f}")
        print(f"Max Loss: ${put_premium:.2f} (premium paid)")
        print(f"Max Gain: ${strike_price - put_premium:.2f}")
        print(f"Current Moneyness: {'ITM' if current_price < strike_price else 'ATM' if abs(current_price - strike_price) < 5 else 'OTM'}")
    else:  # Protective Put
        max_loss_pp = strike_price - current_price - put_premium
        print(f"Strategy: PROTECTIVE PUT (Stock + Put)")
        print(f"Stock Price: ${current_price:.0f}")
        print(f"Put Strike: ${strike_price:.0f}")
        print(f"Put Premium: ${put_premium:.2f}")
        print(f"Max Loss: ${max_loss_pp:.2f}")
        print(f"Max Gain: Unlimited")
        print(f"Insurance Cost: ${put_premium:.2f}")
    print(f"{'='*60}\n")


# Test function
print("✓ Options explorer function defined")

✓ Options explorer function defined


## Interactive Options Explorer

### Adjust the sliders below to explore how parameters affect option profits and losses!

**Instructions:**
- Move **Current Price** to simulate stock price changes
- Adjust **Strike Price** to see ITM, ATM, OTM effects
- Change **Premiums** to see how costs affect profitability
- Select different **Strategies** (Call, Put, or Protective Put)
- Watch the graphs update in real-time!

In [3]:
if widgets_available:
    # Create interactive sliders
    interact(explore_options,
             current_price=FloatSlider(
                 value=100,
                 min=50,
                 max=200,
                 step=5,
                 description='Current Price (S0)',
                 style={'description_width': '180px'},
                 layout={'width': '450px'}
             ),
             strike_price=FloatSlider(
                 value=100,
                 min=50,
                 max=200,
                 step=5,
                 description='Strike Price (K)',
                 style={'description_width': '180px'},
                 layout={'width': '450px'}
             ),
             call_premium=FloatSlider(
                 value=5,
                 min=0.5,
                 max=20,
                 step=0.5,
                 description='Call Premium',
                 style={'description_width': '180px'},
                 layout={'width': '450px'}
             ),
             put_premium=FloatSlider(
                 value=6,
                 min=0.5,
                 max=20,
                 step=0.5,
                 description='Put Premium',
                 style={'description_width': '180px'},
                 layout={'width': '450px'}
             ),
             strategy=Dropdown(
                 options=['Call', 'Put', 'Protective Put'],
                 value='Call',
                 description='Strategy',
                 style={'description_width': '180px'}
             )
    )
else:
    print("Running static example...")
    explore_options(100, 100, 5, 6, 'Call')

interactive(children=(FloatSlider(value=100.0, description='Current Price (S0)', layout=Layout(width='450px'),…

## Guided Experiments: Try These!

### Experiment 1: Call Option Break-even
**Scenario**: Current Price = $100, Strike = $100, Call Premium = $5
- Move current price slider to $110 → See profit start
- Move to $105 → Break-even point
- **Learning**: Profit only when ST > K + Premium

### Experiment 2: Put Option Hedge
**Scenario**: Current Price = $100, Strike = $95, Put Premium = $3
- Move current price down to $80 → Put protects you
- Move current price up to $120 → Put expires worthless
- **Learning**: Put gains value when stock falls

### Experiment 3: Protective Put Insurance
**Select Strategy**: Protective Put
**Scenario**: Current Price = $100, Strike = $95, Put Premium = $5
- Move current price to $70 → Notice max loss = $95 - $100 - $5 = -$10
- Move current price to $150 → Unlimited upside preserved
- **Learning**: Insurance caps downside but costs premium

### Experiment 4: Premium Impact
- Increase Call Premium from $3 to $10 → Break-even shifts right
- Increase Put Premium from $3 to $10 → Max loss increases
- **Learning**: Higher premiums mean need bigger moves to profit

### Experiment 5: Moneyness Comparison
**Select Strategy**: Call
- Strike = $80, Current = $100 → ITM (in the money)
- Strike = $100, Current = $100 → ATM (at the money)
- Strike = $120, Current = $100 → OTM (out of the money)
- **Learning**: Position type determines risk/reward